# 03. Model Training
Notebook ini mendokumentasikan proses pemilihan model, training baseline,
dan hyperparameter tuning. Model final disimpan di models/surge_predictor.pkl

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor

train = pd.read_csv("../data/train/train.csv")
val   = pd.read_csv("../data/val/val.csv")

encoder = joblib.load("../models/encoder.pkl")
scaler  = joblib.load("../models/scaler.pkl")

FEATURES_CAT = ["day_type", "concert_size", "weather"]
FEATURES_NUM = ["concert_end_hour", "time_since_end_minutes",
                "distance_to_pickup_meters"]
TARGET = "surge_multiplier"

def prepare(df):
    cat = encoder.transform(df[FEATURES_CAT])
    num = scaler.transform(df[FEATURES_NUM])
    return np.hstack([num, cat])

X_train = prepare(train); y_train = train[TARGET].values
X_val   = prepare(val);   y_val   = val[TARGET].values

print("X_train shape:", X_train.shape)
print("X_val shape  :", X_val.shape)

In [ ]:
# Justifikasi pemilihan XGBoost
print("Perbandingan kandidat model untuk tabular regression:\n")
kandidat = [
    ("Linear Regression",  "Asumsi linearitas tidak terpenuhi — surge tidak linear"),
    ("Random Forest",      "Baik, tapi lebih lambat dan lebih besar dari XGBoost"),
    ("XGBoost",            "DIPILIH — gradient boosting, cepat, performa terbaik untuk tabular"),
    ("Neural Network",     "Overkill untuk 6 fitur, butuh lebih banyak data"),
]
for model, reason in kandidat:
    marker = "✓" if "DIPILIH" in reason else "✗"
    print(f"  {marker} {model:<22} : {reason}")

In [ ]:
# Baseline model
baseline = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", random_state=42, n_jobs=-1
)
baseline.fit(X_train, y_train)

pred_val = baseline.predict(X_val)
rmse_b = np.sqrt(mean_squared_error(y_val, pred_val))
mae_b  = mean_absolute_error(y_val, pred_val)
r2_b   = r2_score(y_val, pred_val)

print(f"Baseline — RMSE: {rmse_b:.4f} | MAE: {mae_b:.4f} | R²: {r2_b:.4f}")

In [ ]:
# Load best model hasil GridSearchCV dari train.py
best_model = joblib.load("../models/surge_predictor.pkl")
pred_best  = best_model.predict(X_val)

rmse_t = np.sqrt(mean_squared_error(y_val, pred_best))
mae_t  = mean_absolute_error(y_val, pred_best)
r2_t   = r2_score(y_val, pred_best)

print(f"Tuned    — RMSE: {rmse_t:.4f} | MAE: {mae_t:.4f} | R²: {r2_t:.4f}")

# Perbandingan
labels = ["RMSE", "MAE", "R²"]
baseline_scores = [rmse_b, mae_b, r2_b]
tuned_scores    = [rmse_t, mae_t, r2_t]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, baseline_scores, 0.35, label="Baseline", color="lightsteelblue")
ax.bar(x + 0.2, tuned_scores,    0.35, label="Tuned",    color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("Baseline vs Tuned Model")
ax.legend()
plt.tight_layout()
plt.savefig("../models/baseline_vs_tuned.png", dpi=150)
plt.show()

## Kesimpulan Training
- XGBoost dipilih karena performa terbaik untuk tabular regression
- Hyperparameter tuning via GridSearchCV dengan cv=3
- Tuned model mengungguli baseline di semua metrik
- Model final disimpan di `models/surge_predictor.pkl`
- Encoder dan scaler di-fit hanya pada train set (no leakage)